## Checklist general de EDA (para pegar en la pared)

1. Entender el problema y cada fila.
2. Mirada rápida a los datos.
3. Revisar estructura y tipos.
4. Buscar valores faltantes.
5. Buscar duplicados.
6. Analizar variables numéricas.
7. Analizar variables categóricas.
8. Analizar fechas/tiempo (si hay).
9. Analizar texto libre (si hay).
10. Ver relación con la variable objetivo.
11. Buscar correlaciones y multicolinealidad.
12. Buscar outliers y valores raros.
13. Pensar en calidad del dato y “sentido de negocio”.
14. Anotar conclusiones y decisiones (drop, imputar, transformar, etc.).

# Paso 0: Setup básico: Imports & Read data

> (1) **Importar librerías necesarias**  
> (2) **Crear el DataFrame**


(1) **Importar librerías necesarias:**

- pandas para cargar nuestros datos
- sklearn para realizar el ML
- matplotlib y seaborn para graficos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

: 

(2) **Crear el DataFrame:**

In [ ]:
# https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data

df = pd.read_csv("https://raw.githubusercontent.com/4GeeksAcademy/data-preprocessing-project-tutorial/main/AB_NYC_2019.csv")   # o el dataset que uses

# Paso 1: Definición del problema & Información del DataFrame

**Objetivo:** saber qué significa cada fila y qué queremos predecir.
* Cuestiones:

  * ¿Qué representa una fila? ¿Un cliente, una compra, una medición de sensor…?
  * ¿Qué queremos predecir o explicar? → **target** (columna objetivo).
  * ¿Qué columnas solo se conocen después del resultado? (posible leakage)

**Nota:**
Si no saben qué significa una columna → *anotar la duda* y no usarla para modelar hasta entenderla.

> (1) OBSERVACIONES DE LA DATA   
> (2) VARIABLES PREDICTORAS  
> (3) TARGET  
> (4) SELECCIÓN DE COLUMNAS PARA EL ANÁLISIS




###  (1) **OBSERVACIONES DE LA DATA INICIAL**

In [ ]:
df.head()

In [ ]:
df.info()

OBSERVACIONES:    

    - La 1ª columna "#" indica el índex de las variables y la 2ª su nombre/indicador.
    - La 3ª columna cuenta los valores no nulos, es decir dónde sí tenemos data en cada fila. Casi todas las filas contienen todos los valores pero en el caso de las columnas 12 y 13 vemos que faltan unos 10.000 datos.
    - La 4ª columna nos dice el tipo de dato que contienen las variables, numéricos o objeto-string.
    - Por último tenemos un recuento de los tipo de variable y cuánto ocupa en la memoria el total del df.
    - Hay 48895 filas y 16 columnas.

VALORES FALTANTES:         

         - last_review → 38,843 non-null entonces tiene 10,052 valores faltantes

          - reviews_per_month → 38,843 non-null, también tiene 10,052 faltantes

          - name → 48,879 non-null, entonces son 16 faltantes

          - host_name → 48,874 non-null con 21 faltantes

* **Resumen estadístico del df original:**

In [ ]:
df.describe().T.round(2)

In [ ]:
df.describe().T.round(2).sort_values("mean", ascending=False)


OBSERVACIONES:

Al ordenar los valores numéricos más altos en promedio, como id y host_id. Sin embargo, estas variables son identificadores, por lo que su valor promedio no tiene interpretación analítica real, solo reflejan numeración interna.

Las variables realmente relevantes:

        - price (152.7) muestra un promedio considerablemente mayor que su mediana (106), lo que confirma la presencia de valores altos que elevan la media.

        - availability_365 (112.8) indica que, en promedio, los alojamientos están disponibles alrededor de un tercio del año.

        - number_of_reviews (23.3) sugiere que la mayoría de propiedades tienen actividad moderada, aunque con alta dispersión.

        - minimum_nights (7.0) refleja que, en promedio, se piden estancias cortas.

        - calculated_host_listings_count (7.14) tiene una media superior a su mediana (1), lo que indica que existen algunos hosts con muchas propiedades que elevan el promedio.

Se observa alta dispersión y presencia de valores extremos, especialmente en precio, noches mínimas y número de propiedades por anfitrión lo que indica alta presencia de outliers.

### (2) **VARIABLES PREDICTORAS:**

¿Qué representa una fila? ¿Un cliente, una compra, una medición de sensor…? Y ¿cada columa?

> (2.1) Información de las columnas y filas  
> (2.2) Descripción de las variables predictoras:


In [ ]:
df.columns

#### (2.1) **Información de las columnas y filas:**



     - Cada fila indica un alojamiento publicado, las columnas aportan diferente información de cada apartamento.

     - [Link Airbnb Data Dictionary en la version 3 del documento: 'listing csv detail v3']('https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?gid=1322284596#gid=1322284596')

#### (2.2) **Descripción de las variables predictoras:**




1. 'id', ID de la vivienda anunciada.

2. 'name', Nombre del anuncio de la vivienda.

3. 'host_id', ID de Airb&b para el anunciante.

4. 'host_name', Nombre del porpietario de la vivienda.

5. 'neighbourhood_group', Localización: Grupo general del barrio en el qeu se encuentra el apartamento

6. 'neighbourhood', Localización: Vecindario dentro del grupo de barrios

7. 'latitude', Latitud.

8. 'longitude', Longitud.

9. 'room_type', Tipo de habitación.

11. 'minimum_nights', Miinimo de noche disponibles para reservar.

12. 'number_of_reviews', Número total de reseñas del apartamento.

13. 'last_review', Última reseña registrada del apartamento.

14. 'reviews_per_month', Reseñas recividas por mes.

15. 'calculated_host_listings_count', Cuántos apartamentos tiene ofertados un propietario (host).

16. 'availability_365', Disponibilidad anual para alquilar el apartamento en el año 2019. InsideAirbnb: 'Note a listing may not be available because it has been booked by a guest or blocked by the host.'

### (3) **TARGET (OBJETIVO)**

¿Qué queremos predecir o explicar? → **target** (columna objetivo).

> (3.1) Descripción de la variable target  
> (3.2) Estrategia del objetico para le reporte

(3.1) **Descripción de la variable target:**

 - La columna target es el precio ya que buscamos que tipo de apartamento tiene mejor rentabilidad según los datos dados aunque para este análisis no haremos la predicción si no que simplemente se generará un reporte con las conclusiones obtenidas a partir de la data.

 10. 'price', Precio por noche.

        
(3.2) **Estrategia del objetico para le reporte: Por localización y tipo de alojamiento:**

- ¿En qué localización están los barrios que más se alquilan?
- ¿Quétipo de alomajiemnto es el más escogido?
- ¿A qué precio es rentable alquilarlo?


### (4) **SELECCIÓN DE COLUMNAS PARA TRABAJAR EN EL ANÁLISIS:**


* Creamos un nuevo df haciendo una copia de los datos que contiene el original seleccionando solo las columnas que aportan información para comprobar el objetivo.

In [ ]:
columns_analisis = ['id','host_id','name','price','room_type','neighbourhood_group','neighbourhood','minimum_nights','number_of_reviews']
df_analisis = df[columns_analisis].copy()
df_analisis

In [ ]:
df.shape, df_analisis.shape

OBSERVACIONES: 
* En la comaparación de los df, después del filtrado nos quedamos con 9 columnas de 16, estás serán las variables para el análisis.

# Paso 2: Análisis Descriptivo & Limpieza de datos

> (1) NUEVO DataFrame PARA EL ANÁLISIS  
> (2) LIMPIEZA DE DATOS: Eliminar información irrelevante  
>       (2.1) Valores   
>       (2.2) Columnas a 0  
>       (2.3) Valores duplicados


## (1) **NUEVO DataFrame PARA EL ANÁLISIS  :**

In [ ]:
df_analisis.info()

In [ ]:
df_analisis.shape

In [ ]:
df_analisis.head()

## (2) **Limpieza de datos**

Cuando queremos preparar los datos para entrenar un modelo predictivo debemos responder a la siguiente pregunta:

¿Son todas las características imprescindibles para realizar una predicción? Trataremos de hacer es una eliminación controlada de aquellas variables que estamos seguros de que el algoritmo no va a utilizarlas en el proceso predictivo.

Las columnas que utilizaremos en el análisis son:

- Numéricas: 
        
        - 'id', como identificador del anuncio para comprobar duplicados.
        
        - 'host_id', para comprobar duplicados. **
        
        - 'price', como variable predictora. 
        
        - 'availability_365', para revisar valores a 0 y determinar importancia de esta columna comparandola con la información de las reviews.
        
        - 'number_of_reviews', para determinar la fiabilidad del alojamiento anunciado.
        
        - 'minimum_nights', para comparar con los datos de 'number_of_reviews' para determinar si las reviews son del año de la descarga de datos o anteriores.

- Categóricas: 
        
        - 'name', para comprobar duplicados o alores faltantes.

        - 'room_type', para identificar qué tipo de estancia es más rentable alquilar.
        
        - 'neighbourhood_group' y 'neighbourhood', para comparar la agrupación de los apartamentos por localización y saber en qué barrio y vecindario son los más elegidos.

* **Columnas de identificación:**
     - 'id' & 'host_id': Utilizamos el host_id como identificador de cada apartamento.



In [ ]:
df_analisis.set_index('id', drop=True, inplace=True)
df_analisis.head()

### (2.1) **Valores NaNs**

¿Hay valores NaN? ¿Dónde? ¿Representarían información importante para el análisis?  
¿Qué hago con ellos? ¿Decido rellenarlos? ¿Eliminar filas o mantenerlas?

In [ ]:
df_analisis.isna().sum() # Conteo del total de NaNs por columna

* Columna name:

    - Comparación con host_id.    ¿Los apartamentos que tienen el mismo nombre pertenecen al mismo host?

In [ ]:
df_nas = df_analisis.loc[df_analisis["name"].isna()]

* Columna host_id:

In [ ]:
df_analisis['host_id'].isna().sum()

OBSERVACIONES de la columna name & host_id:

     - Hay 4 valores en la columna name que no tienen ningún dato entonces cuatro de los apartamentos publicados aparecen sin el nombre del propietario.
     - Estos apartamentos no pertenecen al mismo propietario ya que ninguno tiene el mismo host_id, todos tienen información distinta por columna.

CONCLUSIÓN:

     - La columna name es un valor que no haría falta utilizar par ala predicción de datos en este caso pero sí aporta información para la conclusión de valores faltantes. 
     - No elimino las filas analizadas.

### (2.2) **Columnas a 0**

* El DataFrame:

In [ ]:
(df_analisis == 0).sum() # Conteo del total de 0 por columna

In [ ]:
df_analisis.loc[(df_analisis == 0).any(axis=1)] # Ver filas que contienen almenos un 0

OBERSAVIONES:

        - De las 10062 filas que contienen valores a 0, solo 11 pertenecen a la columna price y el restante (10052) están en la columna reviews.

CONCLUSIONES:

        - Como los valores a 0 en price son pocas filas decido eliminarlas.
        - También elimino los apartamentos sin review porque los que sí tienen indican con más precisión si el apartamento es funcional para los clientes y rentable para el host. 

In [ ]:
df_analisis = df_analisis[(df_analisis != 0).all(axis=1)]

In [ ]:
(df_analisis == 0).sum()

###  (2.3) **Valores duplicados**

In [ ]:
df_analisis.duplicated().sum()

In [ ]:
df_analisis.apply(lambda x: x.duplicated().sum())


In [ ]:
df_analisis[df_analisis.duplicated]

In [ ]:
df_analisis.sample(5, random_state=0)

In [ ]:
df_analisis.shape

In [ ]:
df_analisis.nunique()

* OBSERVACIONES: 

     - Hay 119 filas duplicadas.
     
     - Los duplicados significativos a analizar principalmente son los de la columna 'name'.

# Paso 3: Análisis de variables

> (1) Descripción de columnas utilizadas para el análisis  
>  
> (2) Análisis de Variables Numéricas, enteros o decimales  
> (2.1) Análisis Univariante  
>   
> (2.1) Análisis Multivariante: Variables Numérica-Numérica  
> (3) Análisis de Variables Categóricas, *tipo de variable que puede tomar uno de un número limitado de categorías o grupo, ninguno de los valores es inherentemente "mayor" o "mejor" que los demás (colores) pero pueden también representarse mediante números finitos (yellow=0, blue=1, pink=2).*  
> (3.1) **Análisis Univariante  
> (3.2) Análisis Multivariante: Variables Categórica - Categórica  
>    
> (4) Análisis Multivariante: Variables Numérico - Categórica  
> (4.1) Distribuciones  
> (4.2) Correlaciones

> * Tipos de análisis: Univariante & Multivariante  
> 1. Univariante: Una variable univariante es un término estadístico que se emplea para referenciar un conjunto de observaciones de un atributo (obervaciones=características, atributo =columna).   
>   - Tipos de variable:
>       1. Numérica - Continua
>       2. Numérica - Discreta
>       3. Categórica - Nominal
>       4. Categórica - Ordinal  
>
> 2. Multivariante: Correlacionamos las variables de mismo tipo entre ellas y con variables numérico-categóricas necesitamos factorizar las categóricas para poder analizar en conjunto de dos tipos de variables distintas.
>   - Análisis egún tipo de variable:
>       1. Numérica - Numérica
>       3. Categórica - Categórica  
>       4. Numérica - Categórica





## (1) **Descripción de columnas utilizadas para el análisis:** 

In [ ]:
col_numericas = df_analisis.select_dtypes(include=['number']).columns.tolist()
col_categoricas = df_analisis.select_dtypes(include=['object']).columns.tolist()

print("Columnas numericas:", col_numericas)
print("Columnas categoricas:", col_categoricas)

Las columnas después de la limpieza de datos:

- Numéricas: 
        
        - 'host_id', para comprobar duplicados. 
        
        - 'price', factor de rentabilidad del analisis. 
        
        - 'number_of_reviews', para determinar la fiabilidad del alojamiento anunciado.
        
        - 'minimum_nights', para saber un promedio de noches que se alquila en la zona.


- Categóricas: 
        
        - 'name', no necesitamos analizar esta variable en esta parte del proceso.

        - 'room_type', para seguir identificando qué tipo de estancia es más rentable alquilar.
        
        - 'neighbourhood_group' y 'neighbourhood', para comparar la agrupación de los apartamentos por localización y saber en qué barrio y vecindario son los más elegidos.

## (2) **Análisis de Variables numéricas**

EN GRÁFICOS 


* NOTA:
**Qué ver:**

    * **Simétrico**: parecido a una campana.
    * **Sesgado a la derecha**: muchos valores pequeños y pocos muy grandes (típico en ingresos, tiempos, montos).
    * **Sesgado a la izquierda**: al revés.

    **Ideas de acciones simples:**

    * Distribución muy sesgada → considerar logaritmo:
      `df["ingreso_log"] = np.log1p(df["ingreso"])`

Transformamos para que los modelos puedan entender mejor. No queremos colas extremas dominantes o escalas desproporcionadas. Aunque estas transformaciones no son necesarias para todos los modelos.


### (2.1) **Análisis Univariante:**

In [ ]:
df_num = df_analisis[col_numericas]
df_num

OBSERVACIONES:

        - Tenemos 4 columnas numéricas y 48895 filas de tipo numérico.

In [ ]:
# GRÁFICOS DE IDEA GENERAL DEL DF:

fig, axis = plt.subplots(3, 2, figsize=(12, 6), gridspec_kw={'height_ratios': [5, 2, 1]})

sns.histplot(ax=axis[0, 0], data=df_num, x="price", kde=True).set(title='Histogramas', xlabel='Price', ylabel='Apartments')
sns.boxplot(ax=axis[0, 1], data=df_analisis, x="price").set(title='Boxplots', ylabel=None)
sns.histplot(ax=axis[1, 0], data=df_analisis, x="minimum_nights", kde=True).set(xlabel='Minimum nights', ylabel='Apartments')
sns.boxplot(ax=axis[1, 1], data=df_analisis, x="minimum_nights")
sns.histplot(ax=axis[2, 0], data=df_analisis, x="neighbourhood_group", kde=True).set(xlabel='Neighbourhood', ylabel='Apartments')
sns.boxplot(ax=axis[2, 1], data=df_analisis, x="neighbourhood_group")

plt.tight_layout()
plt.show()

OBSERVACIONES:

EN HISTOGRAMAS (Con qué frecuencia aparecen los valores):
> * La mayoría de los apartamentos ofrecen un pecio de hasta 1000 por noche aproximadamente
> * El mínimo de noches se centra cerca del 1 y 2 en la mayoria de los anuncios

EN BOXPLOTS (Datos Estadísiticos: Dispersión y Valores atípicos):
> * Las variables con más outliers (valores atípicos) son minimum_nights y price. Divido price en rangos de precios para tratar mejor los datos buscando que tipo de habitación se ajusta a un rango de precio respecto al minimo de noches por reserva.

In [ ]:
figs, axis = plt.subplots(1, figsize=(5, 3))

sns.histplot(ax=axis, data=df_analisis, x='price', kde=True).set(title='Precio', ylabel=None)

In [ ]:
figs, axis = plt.subplots(1, figsize=(5, 3))

sns.histplot(ax=axis, data=df_analisis, x='price', kde=True).set(title='Precio', ylabel=None)
plt.xlim(0,1000)

        - La mayoria de los precios parecen estar concentrados en un rango aproximado de 50 a 100 euros por noche aunque hay varios picos hasta 100 dólares por noche y otros más bajos hasta 400 dólares/noche.

In [ ]:
# PRICE:

df_analisis['price'].describe()

In [ ]:
# Creo una nueva columna categórica del precio dividio por rangos

bins = [0, 60, 100, 190, 600, 1000,10000]
labels = [1, 2, 3, 4, 5, 6]
df_analisis['price_filtered'] = pd.cut(df_analisis['price'], bins=bins, labels=labels, include_lowest=True)

df_analisis['price_filtered'].value_counts().sort_index()

OBSERVACIONES:

> * El mayor número de apartamentos de ofrecen por precios de rango 2 (60-100$) y 3(100-190$)
> * Los rangos 1 (0-60$) y 4(190-600$) contienen un numero de informacion similar.
> * Hay pocos apartaemtnos que se alquilen en el rango de precio 5 (1000-10000$ por noche)

BOXPLOT    - Dentro del rectangulo, dos particiones: cuartil 25% (Q2) y cuartil 50% (Q3)

* **Columna 'number_of_reviews'**:

In [ ]:
df_analisis[df_analisis['number_of_reviews']<60].boxplot(column=['number_of_reviews']) #

In [ ]:
# CÁCLUCO OUTLIERS: 

# FORMULA cálculo quantile 3:  Q3 + 1.5*IQR --> 24 + 1.5(24-1) =~ 60

df_analisis.boxplot(column=['number_of_reviews'])
plt.ylim(0,60) # filtro altura de outliers por rango
# plt.ylim(top=60) # corte del gráfico en 60
plt.show()

In [ ]:
top10_mostreviews = df_analisis.nlargest(10,'number_of_reviews')
top10_mostreviews

In [ ]:
df_analisis['number_of_reviews'].describe()

OBSERVACIONES:  DUDA[[REVISAR - INVESTIGAR - ACABAR]]
- Minimo de reviews encontradas por apartamento en 1
- El máximo son 629 totales.
- La division por cuartiles:  
        - Hasta el 25% de la data contiene solo  de 1 reviews después del filtrado de valores 0.  
        - Del 50% los apartamentos tienen hasta 9 reviews  
        - El siguiente 25% hasta 75% tiene hasta 33 reviews.  
        - En el resto de apartamentos el rango de reviews es muy amplio ya que va desde 40 hasta 629 reviews totales.

### (2.1) **Análisis Multivariante: Variables Numérica-Numérica**

* **CORRELACIONES**

El coeficiente de correlación va de:

> 1  = correlación positiva fuerte  
> 0  = no hay relación  
> -1 = correlación negativa fuerte

In [ ]:
corr_df = df[[ 'price', 'number_of_reviews']].corr()
mask_corr = np.triu(np.ones_like(corr_df, dtype=bool))

fig, axis = plt.subplots(2, 1, figsize=(10, 6))
sns.regplot(ax=axis[0], data=df, x="price", y="number_of_reviews", line_kws={'color': 'red'})
sns.heatmap(ax=axis[1], data=corr_df, mask=mask_corr, annot=True, fmt=".2f")

plt.tight_layout()
plt.show()

OBSERVACIONES:

        - Apartamentos con precios altos señalan que cuánto más caro es el apartamento menos reseñas hay, nulas en algunos casos.
        - La línea roja es ligeramente descendente indicando una correlación negativa (-0.05). 

In [ ]:
# Genero las correlaciones entre la variable target y las predictoras numéricas
corr_price_availability = df_analisis[["price_filtered", "minimum_nights"]].corr()
corr_price_minimum_nights = df_analisis[["price_filtered", "number_of_reviews"]].corr()

# Creo un diagrama de dispersión múltiple
fig, axis = plt.subplots(2, 2, figsize=(7, 7))

sns.regplot(ax=axis[0, 0], data=df_analisis, x="price_filtered", y="number_of_reviews", line_kws={'color': 'green'})
sns.heatmap(ax=axis[1, 0], data=corr_price_minimum_nights, annot=True, fmt=".2f", cbar=False)

sns.regplot(ax=axis[0, 1], data=df_analisis, x="price_filtered", y="minimum_nights", line_kws={'color': 'orange'}).set(ylabel=None)
sns.heatmap(ax=axis[1, 1], data=corr_price_availability, annot=True, fmt=".2f")

plt.tight_layout()
plt.show()

NOTA:

        - Con el objetivo de reducir la influencia de valores extremos y mejorar la interpretabilidad del análisis, usamos la variable price transformada en una variable categórica ordinal, creando intervalos de precio.
        ```
        bins = [0, 60, 100, 190, 600, 1000,10000]
        labels = [1, 2, 3, 4, 5, 6]
        df_analisis['price_filtered'] = pd.cut(df_analisis['price'], bins=bins, labels=labels, include_lowest=True)
        ```

In [ ]:
df_analisis['price_filtered'].value_counts().sort_index()

OBSERVACIONES:

> * Correlación negativa entre precio filtrado y el numéro de reviews no varia mucho de la anterior comparación. 
> * Respecto al minimo de noches, la correlación sigue siendo baja pero no negativa auqnue las dos variables siguen siendo valores que aportarían poco a la predicción.  
> * La mayoría de los apartamentos se encuentran en un rango de precios menor a 1000.  
> * Destacan dos intervalos de precio 100-190 y 190-600 dólares por noche que pertenecen a 23186 filas del conjunto.


## (3) **Análisis de Variables Categóricas:**

### (3.1) **Análisis Univariante:**

In [ ]:
fig, axis = plt.subplots(2, figsize=(7, 7))
sns.histplot(ax=axis[0], data=df, x="room_type").set(ylabel='Apartments')
sns.histplot(ax=axis[1], data=df, x="neighbourhood_group").set(ylabel='Apartments')

plt.tight_layout()
plt.show()

OBSERVACIONES:

> * La mayoria de estancias ofrecidas son apartamentos enteros y habitaciones privadas. Hay poca diferencia entre ellas
> * Los dos barrios donde se forecen más alquileres son Brooklyn y Manhattan, el que menos ofrece es Staten Island

* Características del apartamento:

     - 'room_typ  

In [ ]:
df_analisis['room_type'].value_counts()

In [ ]:
# TOP 10 VECINDARIOS:

fig, axis = plt.subplots(1, figsize = (7, 5))

col = "neighbourhood"
col_neighbourhood = "neighbourhood"
df_analisis[col_neighbourhood].value_counts().head(10).plot(kind="bar")

plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

OBSERVACIONES: 

        - Los dos primeros barrios con más apartamentos son Williamsburg y Bedfor-Stuyvesant con una diferencia de unos mil apartamentos más respecto al resto de vecindarios.

In [ ]:
fig, axis = plt.subplots(2, figsize=(7, 7))
sns.histplot(ax=axis[0], data=df, x="room_type").set(ylabel='Apartments')
sns.histplot(ax=axis[1], data=df, x="neighbourhood_group").set(ylabel='Apartments')

plt.tight_layout()
plt.show()

OBSERVACIONES:

        - La mayor cantidad de apartamentos se encuentran en Manhattan y Brooklyn.
        - El tipo de alquiler predominante es de apartamento entero o habitación privada.

### (3.2) **Análisis Multivariante: Variables Categórica - Categórica**

In [ ]:
fig, axis = plt.subplots(1, figsize = (7, 5))

sns.countplot( data = df_analisis, x = "room_type", hue = "neighbourhood_group")

plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

OBSERVACIONES:

        - En Manhsttan encontramos en gran parte de apartamentos enteros y habitaciones privadas.
        - En Brooklyn parece que encontramos casi el mismo numero de apartamentos enteros y habitaciones privadas.

* **MAPAS:**

In [ ]:
plt.figure(figsize=(10, 8))

# Creamos el mapa de puntos
sns.scatterplot(
    x='longitude',
    y='latitude',
    hue='neighbourhood_group', # Pinta cada punto según su barrio
    data=df,
    alpha=0.6 # Hace los puntos un poco transparentes para ver dónde hay mucha densidad
)

plt.title('Mapa de Alojamientos por Barrio')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.legend(title='Barrio') # Muestra la leyenda de colores
plt.show()

## (4) **Análisis Multivariante: Variables Numérico - Categórica**


* **Ingresos promedio por grupo de barrio:**

In [ ]:
ingr_est_by_group = df.groupby('neighbourhood_group')['price'].mean().sort_values(ascending=False)
print("\nIngresos mensuales promedio estimados por grupo de vecindario:")
print(ingr_est_by_group)

* **Ingresos promedio por tipo de habitacion:**

In [ ]:
ingr_est_by_group = df.groupby('room_type')['price'].mean().sort_values(ascending=False)
print("\nIngresos mensuales promedio estimados por grupo de vecindario:")
print(ingr_est_by_group)

OBSERVACIONES:

        - La media de los preicios por apartamento indica que los barrios más caros son Manhattan, Brooklyn y Staten Island. Los dos últimos solo tienen una diferencia de 10 dólares. En Manhattan el precio medio por noche es de 70 dólares más que Brooklyn.

### (4.1) **Distribuciones:**

* **Precio por rangos vs barrios**

Countplot, cuenta cuántos elementos hay sin importar le precio. En este caso se considera el precio para separar el conteo por rangos.

In [ ]:
fig, axis = plt.subplots(2, figsize=(15, 7))

sns.countplot(ax=axis[0], data=df_analisis, x="neighbourhood_group", hue="price_filtered", palette='rainbow').set(xlabel='BARRIOS', ylabel='Precio')
sns.countplot(ax=axis[1], data=df_analisis, x="room_type", hue="price_filtered", palette='rainbow').set(xlabel='TIPOS DE ESTANCIA', ylabel='Precio')                                   

plt.tight_layout()
plt.show()

OBSERVACIONES: CAMBIAAAAR

> * Se observa que los precios de los partamentos están más altos en Brooklyn y Manhattan mientras que Staten Island y Bronx tienen precios más bajos y menos oferta de estancias también.
> * Los precios más caros son de los aparetamentos enteros y los más económicos de las habitaciones comparticas. Las habitaciones privadas tienen un precio promedio.

CONCLUSÓN:

> El precio sí está relacionado con la ubicación del apartamento y es un factor clave para la rentabilidad del apartamento.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_price_filter2 = df_analisis[df_analisis["price"] < 1000].copy()

fig, axis = plt.subplots(2, 1, figsize=(12, 10))

# Boxplot: Precio por tipo de habitación (coloreado por barrio)
sns.boxplot(ax=axis[0], data=df_price_filter2, x='room_type', y='price', hue='neighbourhood_group', palette='plasma')

axis[0].set_title('Distribución de precios por tipo de habitación y barrio')
axis[0].set_ylabel('Price')
axis[0].set_xlabel('Room Type')

# Boxplot: Precio por barrio (coloreado por tipo de habitación)
sns.boxplot( ax=axis[1], data=df_price_filter2, x='neighbourhood_group', y='price', hue='room_type', palette='plasma')

axis[1].set_title('Distribución de precios por barrio y tipo de habitación')
axis[1].set_ylabel('Price')
axis[1].set_xlabel('Neighbourhood Group')

plt.tight_layout()
plt.show()


OBSERVACIONES:

> * El precio de la mayoria de apartamentos en ese rango se encuentra aproximadamente entre los 50 y 250 dólares por noche.    
> * El tipo de apartamento más predominante en todos los barrios es Entero o habitación privada. En Staten island encontramos también un gran numero de habitaciones compartidas.  
> * Al filtrar el precio por noche a menos de mil vemos que ya existe una gran cantidad de Outliers a partir de los 170 $ por noche aprox. en la mayoría de las distribuciones. 

### (4.2) **Correlaciones:**

* **Factorización de las variables categóricas:**

In [ ]:
# Variables categóricas:
df_encoded =  df_analisis["room_type_n"] = pd.factorize(df_analisis["room_type"])[0]
df_encoded = df_analisis["neighbourhood_group_n"] = pd.factorize(df_analisis["neighbourhood_group"])[0]

* **DIAGRAMA DE DISPERSION**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_price_filter2 = df_analisis[df_analisis["price"] < 1000].copy()

# Factorizamos categorías
df_price_filter2['neighbourhood_group_n'] = pd.factorize(df_price_filter2['neighbourhood_group'])[0]
df_price_filter2['room_type_n'], room_labels = pd.factorize(df_price_filter2['room_type'])

plt.figure(figsize=(10, 6))

# Scatter
sc = plt.scatter(df_price_filter2['neighbourhood_group_n'], df_price_filter2['price'], c=df_price_filter2['room_type_n'], cmap='plasma', alpha=0.6)

# Ajustar ticks y etiquetas eje X
plt.xticks(ticks=range(len(df_price_filter2['neighbourhood_group'].unique())), labels=df_price_filter2['neighbourhood_group'].unique())

plt.xlabel('Neighbourhood Group')
plt.ylabel('Price')
plt.title('Precio por grupo de vecindario (coloreado por tipo de habitación)')
plt.ylim(0, 1000)

# Colorbar pero con etiquetas de room_type
cbar = plt.colorbar(sc)
cbar.set_ticks(range(len(room_labels)))       # posiciones de los ticks
cbar.set_ticklabels(room_labels)              # nombres de los tipos de habitación
cbar.set_label('Tipo de habitación')

plt.tight_layout()
plt.show()


OBSERVACIONES:

> * Variación de precios por vecindario: Los precios son generalmente más altos en Manhattan y más bajos en Bronx, mostrando una clara diferencia según el grupo de vecindario.  
> * Rangos de precios extremos: La mayoría de los alojamientos se concentran por debajo de los 500 dólares y los valores cercanos a 1000 dólares son casos excepcionales.
> * Distribución por tipo de habitación: Los entire homes/apartments tienden a tener precios más altos que los private rooms o shared rooms y se observa cierta superposición de precios entre private rooms y shared rooms, especialmente en Brooklyn y Queens.



* **HEATMAP**

Escojo las variables que más se han ido relacionando con el objetivo.
        

In [ ]:
columns_analisis = ['price', 'minimum_nights', 'room_type_n', 'neighbourhood_group_n']  

corr = df_analisis[columns_analisis].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, axis = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)

plt.title("Mapa de correlaciones")
plt.tight_layout()
plt.show()


**OBSERVACIONES:** 

> * Relación positiva significativa: room_type_n y price presentan una correlación moderada, indicando que el tipo de habitación influye en el precio.  
> * Las varables neighbourhood_group_n y price también muestran una correlación positiva, reflejando que ciertos vecindarios tienden a ser más caros.  
> * Mínima relación: minimum_nights prácticamente no se correlaciona con price, lo que sugiere que la duración mínima de estancia no impacta el precio en este dataset.

## (5) **Reporte**

* **LA ESTRATEGIA PARA LA ALTA RENTABILIDAD SE DIVIDE EN DOS:**

1- Apartamentos de alto Lujo: Precios muy altos (Filtrado: >$1000) con pocas pero constantes reservas (ej. Nolita, Midtown).

2- La mayoría de apartamentos: Precios moderados/bajos (Rango: $100-$200) (ej. Theater District con 58 reseñas/mes).

* **RENTABILIDAD POR BARRIO**

Barrios más rentables (Promedio):

1-Manhattan: ~$172 USD/mes (por listado promedio)

2-Reinas: ~$138 USD/mes

3-Brooklyn: ~$124 USD/mes

NOTA: Aunque el listado #1 está en Brooklyn, en promedio Manhattan es mucho más rentable debido a sus precios base más altos.

* **TIPO DE HABITACION:**

1-Entire home/apt: ~$198 USD/mes. Mucho más rentable por precio y las reservas son más constantes.

2-Habitación privada: ~$93 USD/mes.

3-Habitación compartida: ~$58 USD/mes